[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/03_circuits/03_circuits.ipynb)

# 03 · 电路与 Attribution（toy transformer 上对拍真 patching）

目标：用纯 numpy 搭一个**可完全掌控**的 toy transformer（1 层 2 头 + MLP），在上面做**真 activation patching**（金标准、O(N) 次前向），再实现 **attribution patching**（一阶近似、一次前后向），**对拍**两者——亲眼看到一阶近似在线性节点上几乎精确、在过 softmax 节点上失效。

路线：toy 模型(带缓存) → clean/corrupt + 度量 → 真 activation patching → 有限差分梯度 → attribution patching → **对拍(相关性)** → 失效诊断(线性 vs 过-softmax) → 节点归因排序 → ✏️ 练习 ×4 → 📖 答案 → 🧪 真实电路胶囊。

> 心智模型：**activation patching = 逐个活检(准但慢 O(N))；attribution patching = 一张 X 光看全身(一阶近似，海选)**。
> 用**有限差分**算梯度（数值上诚实、纯 numpy）——真库里是反向传播，数学等价。

## 1 · toy transformer（带激活缓存 = 模拟 hook）

一个 1 层 transformer：2 个 attention head + 1 个 MLP，维度极小（`d=8, T=5, 2 头, V=6`）。
`run(X, patch)` 返回最后位置的 logits 和一个 **cache**（所有可 patch 节点的激活）。`patch` 是 `{节点名: 替换值}`——这就是用字典模拟 TransformerLens 的 hook。

**可 patch 节点**（统一为按 (组件, 位置) 粒度，便于排序/定位）：
- `h{h}_p{t}`：第 h 个 head 在位置 t 的输出，形状 `(d,)`；
- `mlp_p{t}`：MLP 在位置 t 的输出，形状 `(d,)`；
- `scores{h}`：第 h 个 head 的 **pre-softmax 注意力分数**，形状 `(T,T)`——这是**过 softmax** 的节点（失效演示用）。

In [ ]:
import numpy as np
d, T, H, dh, dmlp, V = 8, 5, 2, 4, 16, 6

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

class ToyTransformer:
    def __init__(self, seed=0):
        r = np.random.default_rng(seed)
        self.WQ = r.standard_normal((H, d, dh)) * 0.6
        self.WK = r.standard_normal((H, d, dh)) * 0.6
        self.WV = r.standard_normal((H, d, dh)) * 0.5
        self.WO = r.standard_normal((H, dh, d)) * 0.5
        self.W1 = r.standard_normal((d, dmlp)) * 0.5; self.b1 = r.standard_normal(dmlp) * 0.1
        self.W2 = r.standard_normal((dmlp, d)) * 0.5; self.b2 = r.standard_normal(d) * 0.1
        self.WU = r.standard_normal((d, V)) * 0.5
    def run(self, X, patch=None):
        patch = patch or {}; cache = {}; resid = X.copy()
        head_pos = np.zeros((H, T, d))
        for h in range(H):
            Q = resid @ self.WQ[h]; Kk = resid @ self.WK[h]; Vv = resid @ self.WV[h]
            scores = Q @ Kk.T / np.sqrt(dh)            # [T,T] pre-softmax
            if f'scores{h}' in patch: scores = patch[f'scores{h}']
            cache[f'scores{h}'] = scores
            att = softmax(scores, axis=-1)             # softmax 非线性！
            o = (att @ Vv) @ self.WO[h]                # [T,d] 该 head 输出
            for t in range(T):
                nm = f'h{h}_p{t}'
                if nm in patch: o[t] = patch[nm]
                cache[nm] = o[t].copy()
            head_pos[h] = o
        resid = resid + head_pos.sum(0)               # residual 线性相加
        pre = resid @ self.W1 + self.b1; act = np.maximum(pre, 0.0)
        mlp = act @ self.W2 + self.b2                 # [T,d]
        for t in range(T):
            nm = f'mlp_p{t}'
            if nm in patch: mlp[t] = patch[nm]
            cache[nm] = mlp[t].copy()
        resid = resid + mlp
        cache['resid_final'] = resid
        logits = resid[-1] @ self.WU                  # 最后位置 logits [V]
        cache['logits'] = logits
        return logits, cache

model = ToyTransformer(0)
X0 = np.random.default_rng(1).standard_normal((T, d)) * 0.6
logits, cache = model.run(X0)
print('logits', np.round(logits, 3))
print('cache 节点数', len(cache), '| 示例节点 h0_p4 形状', cache['h0_p4'].shape, '| scores0 形状', cache['scores0'].shape)
assert logits.shape == (V,) and cache['h0_p4'].shape == (d,) and cache['scores0'].shape == (T, T)
# patch 恒等性：把节点 patch 成它自己，输出不变
logits2, _ = model.run(X0, patch={'h0_p4': cache['h0_p4'].copy()})
assert np.allclose(logits, logits2), 'patch 成自身应不改变输出'
print('✅ toy 模型 + 缓存(hook) 就绪；patch 自身=恒等')

## 2 · clean / corrupt 一对输入 + 度量

造一对只在前两个位置不同的输入：`X_clean`（干净）与 `X_corrupt`（破坏关键信息）。
度量 `M(logits) = logit[a] - logit[b]`（logit difference）——电路分析的标准度量。

我们要找的「电路」= 哪些节点因果地把 corrupt 的度量推回 clean。

In [ ]:
a, b = 0, 1                                  # 关注 logit[0]-logit[1]
def metric(logits): return logits[a] - logits[b]

def make_pair(scale=0.5, seed=3):
    r = np.random.default_rng(seed)
    Xc = r.standard_normal((T, d)) * 0.6
    Xk = Xc.copy()
    Xk[0] += r.standard_normal(d) * scale         # 破坏位置0
    Xk[1] += r.standard_normal(d) * scale         # 破坏位置1
    return Xc, Xk

X_clean, X_corrupt = make_pair(scale=0.5, seed=3)
lc, cache_clean = model.run(X_clean)
lk, cache_corrupt = model.run(X_corrupt)
M_clean, M_corrupt = metric(lc), metric(lk)
full_effect = M_clean - M_corrupt               # 全部信息恢复时的度量变化
print(f'M(clean)={M_clean:.3f}  M(corrupt)={M_corrupt:.3f}  全效应(full)={full_effect:.3f}')
assert abs(full_effect) > 0.3, 'clean 与 corrupt 度量应有明显差异(否则无信号)'
print('✅ clean/corrupt 与度量就绪；目标=找出把 corrupt 推回 clean 的节点')

## 3 · 真 activation patching（金标准，O(N) 次前向）

对每个节点 `c`：把 **clean 的激活** patch 进 **corrupt 运行**（denoising 方向），测度量变化：

`effect(c) = M(corrupt 运行, 把 c 换成 clean) − M(corrupt)`

效果越大 → 该节点越因果地携带了关键信息。**每个节点一次前向 → N 个节点 N 次前向**（这就是 O(N) 之痛）。

In [ ]:
def activation_patch(model, X_corrupt, cache_clean, nodes, base):
    '''对每个节点把 clean 激活 patch 进 corrupt 运行，返回 {node: 度量变化}。N 个节点跑 N 次前向。'''
    eff = {}
    for nm in nodes:
        lp, _ = model.run(X_corrupt, patch={nm: cache_clean[nm]})
        eff[nm] = metric(lp) - base
    return eff

comp_nodes = [f'h{h}_p{t}' for h in range(H) for t in range(T)] + [f'mlp_p{t}' for t in range(T)]
print(f'组件节点数 N = {len(comp_nodes)} (=> 真 patching 要 {len(comp_nodes)} 次前向)')
real_eff = activation_patch(model, X_corrupt, cache_clean, comp_nodes, M_corrupt)
top = sorted(comp_nodes, key=lambda n: -abs(real_eff[n]))[:4]
print('真 patching 最重要的 4 个节点:')
for nm in top: print(f'  {nm:8s} effect={real_eff[nm]:+.4f}')
assert max(abs(v) for v in real_eff.values()) > 0.1, '应有节点显著影响度量'
print('✅ 真 activation patching 完成（这是后面 attribution 要逼近的 ground truth）')

## 4 · 度量对激活的梯度（有限差分）

attribution patching 需要 `∇M`（度量对每个激活的梯度）。真库用**反向传播**一次拿到所有梯度；
这里用**有限差分**算（数值上诚实、纯 numpy、数学等价）：`∂M/∂a_i ≈ (M(a+ε e_i) − M(a)) / ε`，在 **corrupt 运行**处求值。

> 真实工程里这一步是 `loss.backward()`，一次反向得到所有节点梯度（O(1)）；有限差分只是为了在 numpy 里看清它在算什么。

In [ ]:
def grad_metric_wrt(model, X_corrupt, cache_corrupt, nm, base, eps=1e-6):
    '''度量对节点 nm 激活的梯度(有限差分)，在 corrupt 运行处求值。返回与该激活同形状的梯度。'''
    shp = np.asarray(cache_corrupt[nm]).shape
    flat = np.asarray(cache_corrupt[nm], dtype=float).ravel()
    g = np.zeros(flat.size)
    for i in range(flat.size):
        pert = flat.copy(); pert[i] += eps
        lp, _ = model.run(X_corrupt, patch={nm: pert.reshape(shp)})
        g[i] = (metric(lp) - base) / eps
    return g.reshape(shp)

# 验证梯度方向正确：sanity——对一个线性节点，沿梯度走 δ 应让度量增加 ~ δ·‖g‖²
g_demo = grad_metric_wrt(model, X_corrupt, cache_corrupt, 'h0_p4', M_corrupt)
delta = 1e-3
lp, _ = model.run(X_corrupt, patch={'h0_p4': cache_corrupt['h0_p4'] + delta * g_demo})
pred = delta * (g_demo ** 2).sum()
got = metric(lp) - M_corrupt
print(f'沿梯度走 δ={delta}: 预测Δ={pred:.3e}  实际Δ={got:.3e}')
assert np.sign(pred) == np.sign(got) and abs(pred - got) < 0.1 * abs(pred) + 1e-9
print('✅ 有限差分梯度正确（沿梯度走，度量按预测变化）')

## 5 · attribution patching + 对拍真 patching（核心 payoff）

一阶近似：`effect(c) ≈ (a_clean − a_corrupt) · ∇M|corrupt`（逐元素相乘求和）。
**一次 clean 前向 + 一次 corrupt 前向 + 每节点一次梯度**（真库里梯度是一次反向，对所有节点 O(1)）。

然后**对拍**：attribution 估计 vs 真 patching，算相关系数——这是 attribution patching 之所以有用的关键证据。

In [ ]:
def attribution_patch(model, X_corrupt, cache_clean, cache_corrupt, nodes, base):
    '''一阶估计每个节点的 patching 效果：(a_clean - a_corrupt)·grad。'''
    est = {}
    for nm in nodes:
        g = grad_metric_wrt(model, X_corrupt, cache_corrupt, nm, base)
        diff = cache_clean[nm] - cache_corrupt[nm]
        est[nm] = float((diff * g).sum())
    return est

attr_eff = attribution_patch(model, X_corrupt, cache_clean, cache_corrupt, comp_nodes, M_corrupt)
rv = np.array([real_eff[n] for n in comp_nodes])
av = np.array([attr_eff[n] for n in comp_nodes])
corr = np.corrcoef(rv, av)[0, 1]
print(f'attribution vs 真 patching 相关系数 = {corr:.4f}  (越接近1越好)')
print('几个节点对照 (real | attr):')
for nm in sorted(comp_nodes, key=lambda n: -abs(real_eff[n]))[:4]:
    print(f'  {nm:8s} real={real_eff[nm]:+.4f}  attr={attr_eff[nm]:+.4f}')
assert corr > 0.95, 'attribution 应高度相关于真 patching(组件节点路径近线性)'
print('✅ 对拍成功：一次前后向的 attribution 高度逼近 N 次前向的真 patching')

## 6 · 失效诊断：线性节点 vs 过-softmax 节点

为什么上面相关这么高？因为组件输出节点（head/mlp out）到 logit 的路径**近线性**。

对比 **过-softmax 节点**（`scores{h}`，pre-softmax 分数）：它的改动要先过 softmax 非线性。我们在**小扰动**和**大扰动**两种 corrupt 下，比较线性节点与过-softmax 节点的一阶近似**相对误差**。

In [ ]:
def rel_err(model, scale, seed, lin_nodes, sm_nodes):
    Xc, Xk = make_pair(scale=scale, seed=seed)
    lc, cc = model.run(Xc); lk, ck = model.run(Xk); base = metric(lk)
    real = activation_patch(model, Xk, cc, lin_nodes + sm_nodes, base)
    attr = attribution_patch(model, Xk, cc, ck, lin_nodes + sm_nodes, base)
    def mre(ns): return np.mean([abs(attr[n]-real[n])/(abs(real[n])+1e-9) for n in ns])
    return mre(lin_nodes), mre(sm_nodes)

lin = ['h0_p4', 'h1_p4', 'mlp_p4']           # 线性流向 logit
sm = ['scores0', 'scores1']                  # 过 softmax
for scale in [0.25, 1.5]:
    e_lin, e_sm = rel_err(model, scale, 1, lin, sm)
    print(f'扰动 scale={scale}: 线性节点相对误差={e_lin:.3f}  过-softmax节点相对误差={e_sm:.3f}')
e_lin_s, e_sm_s = rel_err(model, 0.25, 1, lin, sm)
e_lin_l, e_sm_l = rel_err(model, 1.5, 1, lin, sm)
assert e_lin_l < 0.1, '线性节点即使大扰动一阶也准'
assert e_sm_l > e_lin_l * 3, '过-softmax 节点的一阶误差应明显更大'
assert e_sm_l > e_sm_s, '扰动越大，过-softmax 节点的一阶近似越差'
print('✅ 复现 AtP 的失效模式：一阶在线性节点≈精确，在 softmax 处随扰动增大而失效 → 需 AtP*')

## 7 · 节点归因排序：定位电路

用 attribution 分数给所有节点排序，看它和真 patching 排序的**重合度**——这是 attribution patching 的实际用法：**飞快海选重要节点**。

In [ ]:
order_real = sorted(comp_nodes, key=lambda n: -abs(real_eff[n]))
order_attr = sorted(comp_nodes, key=lambda n: -abs(attr_eff[n]))
k = 5
overlap = len(set(order_real[:k]) & set(order_attr[:k]))
print(f'真 patching top{k}: {order_real[:k]}')
print(f'attribution top{k}: {order_attr[:k]}')
print(f'top{k} 重合 = {overlap}/{k}')
assert overlap >= 4, 'attribution 的 top-k 应与真 patching 高度重合(海选有效)'
print('✅ attribution 排序≈真 patching 排序：用 1 次前后向就能海选出电路的关键节点')

---
## ✏️ 练习 1：归因近似误差（相关系数）

实现 `atp_correlation(model, scale, seed)`：在给定扰动下，对所有 `comp_nodes` 算真 patching 与 attribution，返回二者的相关系数。复用 `activation_patch` / `attribution_patch`。

验证：小扰动下相关系数很高（>0.9）。

In [ ]:
def atp_correlation(model, scale, seed):
    # TODO: make_pair -> 两次 run 得 cache_clean/cache_corrupt 与 base=M_corrupt
    #   real = activation_patch(...); attr = attribution_patch(...) 都用 comp_nodes
    #   返回 np.corrcoef([real...],[attr...])[0,1]
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
corr_small = atp_correlation(model, 0.3, seed=7)
print(f'小扰动 scale=0.3: attribution vs real 相关={corr_small:.4f}')
assert corr_small > 0.9, '小扰动下一阶近似应高度相关'
print('✅ 练习 1 通过：attribution patching 在小扰动下高保真')

## ✏️ 练习 2：节点归因 top-k 定位

实现 `topk_circuit(eff_dict, k)`：返回按 **|效果|** 降序的前 k 个节点名（list）。

用它分别对真 patching 和 attribution 取 top-3，验证二者重合 ≥2（attribution 能海选出电路节点）。

In [ ]:
def topk_circuit(eff_dict, k):
    # TODO: 返回 eff_dict 中 |value| 最大的 k 个 key（list, 降序）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ck_real = topk_circuit(real_eff, 3)
ck_attr = topk_circuit(attr_eff, 3)
print('真 patching top3:', ck_real)
print('attribution top3:', ck_attr)
assert len(ck_real) == 3 and len(set(ck_attr) & set(ck_real)) >= 2
print('✅ 练习 2 通过：attribution top-k 命中电路关键节点')

## ✏️ 练习 3：边归因（一条 u→v 边）

**边归因**估计「上游节点 u 经由下游节点 v」这条连接的效果：把 u patch 成 clean，但**只测它对 v 输入的改变 × v 的下游梯度**。

这里做一个可计算的玩具版：边 `(u→logits)` 的直接贡献 = `(a_clean[u]−a_corrupt[u]) · ∂M/∂a_u`（即把每个节点直接连到输出）。
实现 `edge_to_output(model, X_corrupt, cache_clean, cache_corrupt, u, base)` 返回这条边的一阶效果（标量）。
（这正是节点 attribution 的单节点形式——边归因的最简特例：下游 v = 输出。）

In [ ]:
def edge_to_output(model, X_corrupt, cache_clean, cache_corrupt, u, base):
    # TODO: g = grad_metric_wrt(...对 u...); diff = cache_clean[u]-cache_corrupt[u]
    #   返回 float((diff*g).sum())
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
e = edge_to_output(model, X_corrupt, cache_clean, cache_corrupt, 'h0_p4', M_corrupt)
print(f"边 (h0_p4 -> output) 一阶效果 = {e:+.4f}")
# 应与该节点的 attribution 分数一致（边到输出 == 节点 attribution）
assert abs(e - attr_eff['h0_p4']) < 1e-6, '边到输出应等于该节点的 attribution'
print('✅ 练习 3 通过：边归因的最简特例(边到输出)=节点归因')

## ✏️ 练习 4：电路定位（最小节点集恢复效应）

贪心地按 attribution 排序逐个加入节点，把这些节点 patch 成 clean，测真 patching 恢复了**全效应**的多少比例。

实现 `recovered_fraction(model, X_corrupt, cache_clean, nodes, base, full)`：返回把 `nodes` 全部 patch 成 clean 后的 `(M−base)/full`。
验证：top-3 attribution 节点恢复的比例，**远高于**随机 3 个节点。

In [ ]:
def recovered_fraction(model, X_corrupt, cache_clean, nodes, base, full):
    # TODO: 一次性把 nodes 全 patch 成 clean，run，返回 (metric(logits)-base)/full
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
order_attr = sorted(comp_nodes, key=lambda n: -abs(attr_eff[n]))
rec_top3 = recovered_fraction(model, X_corrupt, cache_clean, order_attr[:3], M_corrupt, full_effect)
rng4 = np.random.default_rng(0)
rec_rand = np.mean([
    recovered_fraction(model, X_corrupt, cache_clean,
                       list(rng4.choice(comp_nodes, 3, replace=False)), M_corrupt, full_effect)
    for _ in range(20)])
print(f'top3 attribution 节点恢复 {rec_top3:.2f} of full | 随机3节点恢复 {rec_rand:.2f}')
assert rec_top3 > rec_rand + 0.3, 'attribution 选出的少数节点应恢复远多于随机'
print('✅ 练习 4 通过：少数 attribution 节点=紧凑电路(恢复大部分效应)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def atp_correlation(model, scale, seed):
    Xc, Xk = make_pair(scale=scale, seed=seed)
    _, cc = model.run(Xc); lk, ck = model.run(Xk); base = metric(lk)
    real = activation_patch(model, Xk, cc, comp_nodes, base)
    attr = attribution_patch(model, Xk, cc, ck, comp_nodes, base)
    rv = np.array([real[n] for n in comp_nodes]); av = np.array([attr[n] for n in comp_nodes])
    return np.corrcoef(rv, av)[0, 1]

In [ ]:
# 练习 2 参考答案
def topk_circuit(eff_dict, k):
    return sorted(eff_dict, key=lambda n: -abs(eff_dict[n]))[:k]

In [ ]:
# 练习 3 参考答案
def edge_to_output(model, X_corrupt, cache_clean, cache_corrupt, u, base):
    g = grad_metric_wrt(model, X_corrupt, cache_corrupt, u, base)
    diff = cache_clean[u] - cache_corrupt[u]
    return float((diff * g).sum())

In [ ]:
# 练习 4 参考答案
def recovered_fraction(model, X_corrupt, cache_clean, nodes, base, full):
    lp, _ = model.run(X_corrupt, patch={n: cache_clean[n] for n in nodes})
    return (metric(lp) - base) / full

---
## 🧪 真实数据胶囊：IOI 电路与归因的规模账

用真实数字体会 attribution patching 的价值。**GPT-2 small** 有 12 层 × 12 head = **144 个 attention head**；Wang 2022 的 **IOI 电路**最终只用了约 **26 个 head**（分成 name mover / S-inhibition / induction / duplicate token 等几类）。

下面算：在不同粒度下，真 activation patching 要跑多少次前向，attribution patching 又是多少。

In [ ]:
# GPT-2 small 真实规模
n_layers, n_heads = 12, 12
d_model, d_mlp, seq = 768, 3072, 128
n_head_nodes = n_layers * n_heads                       # 144 个 head
n_neuron_nodes = n_layers * d_mlp                       # MLP 神经元
n_headpos_nodes = n_head_nodes * seq                    # (head, 位置) 粒度
ioi_circuit_heads = 26                                  # Wang 2022 最终电路

print(f'GPT-2 small: {n_head_nodes} head, {n_neuron_nodes:,} MLP神经元')
print(f'{"粒度":<22}{"真 patching 前向次数":>20}{"attribution 前后向":>20}')
for name, N in [('head 级', n_head_nodes), ('(head,位置) 级', n_headpos_nodes), ('neuron 级', n_neuron_nodes)]:
    print(f'{name:<22}{N:>20,}{"~2 (与 N 无关)":>20}')
speedup = n_neuron_nodes / 2
print(f'\nneuron 级加速比 ≈ {speedup:,.0f}x  —— 这就是 attribution patching 让前沿电路发现可行的原因')
assert n_head_nodes == 144 and speedup > 1000
print(f'IOI 电路只占 {ioi_circuit_heads}/{n_head_nodes} = {ioi_circuit_heads/n_head_nodes:.0%} 的 head -> 电路是稀疏的')

**🧪 胶囊练习**：实现 `forward_passes(N, method)`：`method='activation'` 返回 `N`（逐个 patch），`method='attribution'` 返回 `2`（一前一后，与 N 无关）。用它算 neuron 级（N=36864）两种方法的前向次数比。

In [ ]:
def forward_passes(N, method):
    # TODO: 'activation' -> N; 'attribution' -> 2; 其它 raise ValueError
    raise NotImplementedError

In [ ]:
# 自测
N = 12 * 3072
ap = forward_passes(N, 'activation'); at = forward_passes(N, 'attribution')
assert ap == N and at == 2
print(f'neuron 级: activation={ap:,} 次前向, attribution={at} 次 -> 加速 {ap/at:,.0f}x')
print('✅ 胶囊练习通过：attribution 把 O(N) 压成 O(1)')

In [ ]:
# 📖 胶囊参考答案
def forward_passes(N, method):
    if method == 'activation': return N
    if method == 'attribution': return 2
    raise ValueError(method)

### 小结
- **电路 = 行为背后的因果子图**（节点=组件/特征，边=信息流）；找电路 = 因果定位。
- **activation patching**：金标准，把 clean 激活 patch 进 corrupt 测恢复；但要 **O(N) 次前向**。
- **attribution patching**：一阶近似 `Δa·∇M`，**一次前后向归因所有节点**，O(N)→O(1)；是**海选器**不是判决书。
- **失效**：一阶在线性节点≈精确，**过 softmax 节点**随扰动增大而失效 → **AtP\*** 修正。
- **边归因(EAP)** 给出连接结构；**DLA** 读输出端；**稀疏特征电路(Marks 2024)** 把节点从 head 升级为可命名的 SAE 特征。

下一站：**模块 04 · 特征 Steering** —— 读懂电路与特征之后，怎么定向、干净地操控模型行为。